# dscraft.eda quickstart

This notebook demonstrates `dscraft.eda`'s single entry point, `LazyEDA`, on a small synthetic `polars.DataFrame`: schema/null profiling, HLL cardinality and KLL quantile sketches, a mixed-type association matrix, and a self-contained HTML/Canvas report export.

The synthetic dataset deliberately mixes column shapes -- a numeric column with nulls, a low-cardinality categorical column, a high-cardinality "id"-style string column, and a boolean column -- to exercise every column-category branch `LazyEDA.profile` supports.

This mirrors `packages/dscraft/examples/eda/profile_example.py` in notebook form.

Requires the `eda` extra:
```bash
pip install -e "packages/dscraft[eda]"
```

In [1]:
import tempfile
from pathlib import Path

import polars as pl

from dscraft.eda import LazyEDA

ROWS = 200

countries = ["US", "US", "US", "CA", "CA", "MX", "DE", "FR"]
ages = [float(20 + (i * 7) % 60) for i in range(ROWS)]
# Sprinkle in a few nulls so the null-percentage summary has something to report.
ages = [None if i % 37 == 0 else age for i, age in enumerate(ages)]

df = pl.DataFrame(
    {
        "customer_id": [f"cust-{i:06d}" for i in range(ROWS)],  # high-cardinality string
        "country": [countries[i % len(countries)] for i in range(ROWS)],  # low-cardinality
        "age": ages,  # numeric, with nulls
        "is_subscribed": [i % 3 == 0 for i in range(ROWS)],  # boolean
    }
)

print(f"Profiling a synthetic {df.shape[0]}-row x {df.shape[1]}-column dataset...")
profile = LazyEDA().profile(df, title="Synthetic Customer Dataset EDA")

print(f"Row count: {profile.row_count}")
print(f"Columns with at least one null: {profile.null_report.columns_with_nulls()}")

Profiling a synthetic 200-row x 4-column dataset...
Row count: 200
Columns with at least one null: ['age']


## Per-column summary

In [2]:
for column in profile.schema_report.columns:
    name = column.name
    null_pct = profile.null_report.null_percentages[name]
    line = f"  {name:15s} category={column.category:9s} null%={null_pct:5.1f}"
    if name in profile.quantile_results:
        median = profile.quantile_results[name].quantile_estimates[0.5]
        line += f"  median~={median:.1f}"
    if name in profile.cardinality_results:
        estimate = profile.cardinality_results[name].estimate
        line += f"  distinct~={estimate:.0f}"
    print(line)

if profile.association_matrix is not None:
    print()
    print(f"Association matrix computed across columns: {profile.association_matrix.columns}")
    if profile.association_matrix.unavailable_pairs:
        print(f"  Unavailable pairs: {list(profile.association_matrix.unavailable_pairs)}")

  customer_id     category=string    null%=  0.0  distinct~=200
  country         category=string    null%=  0.0  distinct~=5
  age             category=numeric   null%=  3.0  median~=49.0
  is_subscribed   category=boolean   null%=  0.0

Association matrix computed across columns: ['customer_id', 'country', 'age', 'is_subscribed']


## Export a self-contained HTML report

In [3]:
output_path = Path(tempfile.gettempdir()) / "dscraft_eda_quickstart_report.html"
profile.export(output_path)
print(f"Wrote self-contained HTML EDA report to: {output_path}")
print(f"Report file size: {output_path.stat().st_size} bytes")
print("Open that file directly in a browser to view it (no server needed).")

Wrote self-contained HTML EDA report to: /var/folders/bw/9966jv5s41zgvrvx6hpss6bh0000gn/T/dscraft_eda_quickstart_report.html
Report file size: 9990 bytes
Open that file directly in a browser to view it (no server needed).


## Summary

This notebook profiled a small, deliberately mixed-dtype synthetic dataset with `dscraft.eda.LazyEDA`: schema and null reports, HLL cardinality estimates for string columns, KLL quantile estimates for numeric columns, a mixed-type association matrix, and a self-contained HTML report export.

See `packages/dscraft/README.md`'s `## dscraft.eda` section for more detail on `LazyEDA`'s column-routing heuristic and what's deferred.